In [ ]:
import pandas as pd
import missingno as msno
import geopandas as gpd
from shapely.geometry import Point
import rasterio
import matplotlib.pyplot as plt
from rasterio.plot import show
import numpy as np
from affine import Affine
from rasterio.transform import from_origin

## Merge RegBL and Radon Map

This notebooks' objective is to merge two datasets in their `processed` versions: 

- `02_OFSP_radon_map`
- `03_OFS_regbl`

## Constants

In [ ]:
ROOT = '../../'

## Load the data

### Radon threshold exceedance probability

In [ ]:
src = rasterio.open(f'{ROOT}data/raw/02_OFSP_radon_map/Radon_carte_probabilite.tif')

radon_prob_data = src.read(1)

print(src.profile)
print(src.bounds)
print(src.crs)

In [ ]:
transform = from_origin(2480475.0400290643, 1301499.9458016460, 1000, 1000)

with rasterio.open(f"{ROOT}data/raw/02_OFSP_radon_map/Radon_carte_probabilite.tif") as src:
    radon_prob_data = src.read(1)

    profile = src.profile
    profile.update({
        "transform": transform,
        "width": 345,
        "height": 250,
        "nodata": -1.7e+308
    })

    with rasterio.open(f"{ROOT}data/raw/02_OFSP_radon_map/Radon_carte_probabilite.tif", "w", **profile) as dst:
        dst.write(radon_prob_data, 1)

In [ ]:
src = rasterio.open(f'{ROOT}data/raw/02_OFSP_radon_map/Radon_carte_probabilite.tif')

radon_prob_data = src.read(1)

In [ ]:
radon_prob_data = np.ma.masked_where(radon_prob_data == -1.7e+308, radon_prob_data)
fig, ax = plt.subplots(figsize=(10, 10))
cax = ax.imshow(radon_prob_data, cmap='viridis')
plt.show()

### Confidence index of radon threshold exceedance probability

In [ ]:
src = rasterio.open(f'{ROOT}data/raw/02_OFSP_radon_map/Radon_carte_distance.tif')

radon_dist_data = src.read(1)

print(src.profile)
print(src.bounds)
print(src.crs)

In [ ]:
transform = from_origin(2480475.0400290643, 1301499.9458016460, 1000, 1000)

with rasterio.open(f"{ROOT}data/raw/02_OFSP_radon_map/Radon_carte_distance.tif") as src:
    radon_dist_data = src.read(1)

    profile = src.profile
    profile.update({
        "transform": transform,
        "width": 345,
        "height": 250,
        "nodata": -1.7e+308
    })

    with rasterio.open(f"{ROOT}data/raw/02_OFSP_radon_map/Radon_carte_distance.tif", "w", **profile) as dst:
        dst.write(radon_dist_data, 1)

In [ ]:
src = rasterio.open(f'{ROOT}data/raw/02_OFSP_radon_map/Radon_carte_distance.tif')

radon_dist_data = src.read(1)

In [ ]:
radon_dist_data = np.ma.masked_where(radon_dist_data == -1.7e+308, radon_dist_data)
fig, ax = plt.subplots(figsize=(10, 10))
cax = ax.imshow(radon_dist_data, cmap='viridis')
plt.show()

### Current dataset

In [ ]:
df_regbl = pd.read_csv(f'{ROOT}data/processed/dataset-v0.6.csv', index_col=0)

df_regbl.head()

## Convert main DataFrame to GeoDataFrame

In [ ]:
df_regbl = gpd.GeoDataFrame(df_regbl, geometry=gpd.points_from_xy(df_regbl["Coordonnée E"], df_regbl["Coordonnée N"]), crs='EPSG:2056')

## Overlay radon threshold exceedance probability raster on RegBL

In [ ]:
with rasterio.open(f'{ROOT}data/raw/02_OFSP_radon_map/Radon_carte_probabilite.tif') as src:
    radon_prob_band = src.read(1)
    raster_crs = src.crs
    transform = src.transform

    radon_prob_values = []
    for geom in df_regbl.geometry:
        try:
            row, col = src.index(geom.x, geom.y)
            val = radon_prob_band[row, col]
        except IndexError:
            val = None
        radon_prob_values.append(val)

df_regbl["Rdn Map - Probabilité dépassement valeur réf radon"] = radon_prob_values

## Overlay radon threshold exceedance probability confidence index raster on RegBL

In [ ]:
with rasterio.open(f'{ROOT}data/raw/02_OFSP_radon_map/Radon_carte_distance.tif') as src:
    radon_dist_band = src.read(1)
    raster_crs = src.crs
    transform = src.transform

    radon_dist_values = []
    for geom in df_regbl.geometry:
        try:
            row, col = src.index(geom.x, geom.y)
            val = radon_dist_band[row, col]
        except IndexError:
            val = None
        radon_dist_values.append(val)

df_regbl["Rdn Map - Index de confiance dépassement valeur réf"] = radon_dist_values

In [ ]:
df_regbl.head()

In [ ]:
df_regbl.shape

In [ ]:
msno.matrix(df_regbl)

## Save the data

In [ ]:
df_regbl.to_csv(f"{ROOT}data/processed/dataset-v0.8.csv")